# 🎵 Music Recommendation System

A simple music recommendation system built using Spotify data.

The goal of this project is to recommend songs that are similar to a song the user selects, based on its audio features, genre, and other information.

## How It Works

The basic idea is pretty simple:

1. Load and clean the Spotify dataset.
2. Select the features that are useful for finding similar songs.
3. Scale the numerical features and encode the genres.
4. Combine everything into one feature matrix.
5. Find songs that are most similar using cosine similarity.
6. Use genre and artist information to improve the recommendations.
7. Return the most similar songs to the user.



# 🎵 Music Recommendation System

This project builds a content-based music recommendation system using Spotify track metadata and audio features.

The system recommends songs based on their similarity in musical characteristics such as danceability, energy, acousticness, valence, and genre.

In [1]:
import pandas as pd
import numpy as np


## 1. Load the Dataset

First, let's load the Spotify dataset and take a look at what we're working with.



In [2]:
songs = pd.read_csv('spotify-tracks-dataset-detailed.csv')
songs.head()


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## 2. Data Selection & Cleaning

There are quite a few columns in the original dataset, so I'll keep the ones that are useful for the recommendation system and remove missing values.



In [3]:
songs = songs[[
    'track_name',
    'artists',
    'track_genre',
    'danceability',
    'popularity',
    'energy',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence'
]]

songs = songs.dropna().drop_duplicates().reset_index(drop=True)

print('Rows:', len(songs))
print('Columns:', len(songs.columns))


Rows: 106946
Columns: 11


## 3. Remove Duplicate Tracks

The dataset contains duplicate tracks, so I'll remove them to avoid getting the same song multiple times in the recommendations.



In [4]:
songs = songs.drop_duplicates(
    subset=['track_name', 'artists']
).reset_index(drop=True)

print('Unique songs:', len(songs))


Unique songs: 81343


## 4. Normalize Song Titles

I'll clean up the song titles a little by converting them to lowercase and removing the extra text inside parentheses.



In [5]:
songs['clean_title'] = (
    songs['track_name']
    .str.lower()
    .str.split('(')
    .str[0]
    .str.strip()
)


## 5. Select Audio Features

Now I'll select the audio features that will be used to compare songs with each other.



In [6]:
numerical_features = [
    'danceability',
    'popularity',
    'energy',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence'
]

X_numeric = songs[numerical_features]


## 6. Feature Scaling

The audio features have different ranges, so I'll scale them before using them for similarity calculations.



In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(X_numeric)


## 7. Encode Music Genres

Since genre is a categorical feature, I'll convert it into numerical values using one-hot encoding.



In [8]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown='ignore')
X_genre = encoder.fit_transform(songs[['track_genre']])


## 8. Build the Recommendation Feature Matrix

Now I'll combine the scaled audio features and encoded genres into one feature matrix. This will be the data used to find similar songs.



In [9]:
from scipy.sparse import hstack, csr_matrix

X = hstack([
    csr_matrix(X_numeric_scaled),
    X_genre * 0.3
])

print('Songs:', len(songs))
print('Feature matrix:', X.shape)


Songs: 81343
Feature matrix: (81343, 121)


## 9. Recommendation Engine

Now comes the main part of the project.

I'll use cosine similarity to find songs that are closest to the selected song based on their features.



In [10]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_song(song_name, n=10):

    # Find the song
    matches = songs[
        songs['track_name'].str.lower() == song_name.lower()
    ]

    if matches.empty:
        return "Song not found."

    index = matches.index[0]

    # Input song details
    input_genre = songs.loc[index, 'track_genre']
    input_title = songs.loc[index, 'clean_title']
    input_artists = str(
        songs.loc[index, 'artists']
    ).split(';')

    # Similarity
    similarity = cosine_similarity(
        X[index], X
    ).flatten()

    results = songs.copy()
    results['similarity'] = similarity

    # Genre match
    results['genre_match'] = (
        results['track_genre'] == input_genre
    ).astype(float)

    # Artist match
    def same_artist(artists):
        if pd.isna(artists):
            return False

        return any(
            artist in str(artists).split(';')
            for artist in input_artists
        )

    results['artist_match'] = (
        results['artists']
        .apply(same_artist)
        .astype(float)
    )

    # Final score
    results['score'] = (
        results['similarity'] * 0.85
        + results['genre_match'] * 0.10
        + results['artist_match'] * 0.05
    )

    # Remove original song
    results = results[
        results.index != index
    ]

    # Remove same song / alternate versions
    results = results[
        results['clean_title'] != input_title
    ]

    # Sort
    results = results.sort_values(
        'score',
        ascending=False
    )

    # Remove duplicate versions
    results = results.drop_duplicates(
        subset='clean_title'
    )

    # Return clean result
    return results.head(n)[[
        'track_name',
        'artists',
        'track_genre',
        'score'
    ]].reset_index(drop=True)
  

In [11]:
recommend_song('Tum Hi Ho', 10)


,track_name,artists,track_genre,score
0,"Rait Zara Si (From ""Atrangi Re"")",A.R. Rahman;Arijit Singh;Shashaa Tirupati,pop-film,0.988924
1,Fitoor,Mithoon;Arijit Singh;Neeti Mohan;Karan Malhotra,pop-film,0.988911
2,Humdard,Arijit Singh,pop-film,0.984665
3,Phir Bhi Tumko Chaahunga,Mithoon;Arijit Singh;Shashaa Tirupati,pop-film,0.983670
4,Khamoshiyan,Jeet Gannguli;Arijit Singh,pop-film,0.983529
5,"Baatein Ye Kabhi Na (From ""Khamoshiyan"") - Male",Jeet Gannguli;Arijit Singh,pop-film,0.982488
6,Phir Mohabbat,Mohammed Irfan;Arijit Singh;Saim Bhat,pop-film,0.982195
7,Aasan Nahin Yahan,Arijit Singh,pop-film,0.982134
8,"Tujhe Kitna Chahne Lage (From ""Kabir Singh"")",Arijit Singh;Mithoon,pop-film,0.977881
9,Baatein Ye Kabhi Na - Male,Jeet Gannguli;Arijit Singh,pop-film,0.973465


In [12]:
recommend_song('Good to Me', 10)


,track_name,artists,track_genre,score
0,Believer,Friction,drum-and-bass,0.978470
1,By Your Side,Friction;Flowidus;Raphaella,drum-and-bass,0.973441
2,Remember,Friction,drum-and-bass,0.959542
3,I Need To Feel,Friction;Poppy Baskcomb,drum-and-bass,0.954417
4,Bad To Me (feat. Grace Grundy),Hybrid Minds;Grace Grundy,drum-and-bass,0.946987
5,Heaven,Delta Heavy;Jem Cooke,drum-and-bass,0.942926
6,Gravity,Metrik,drum-and-bass,0.941987
7,Nobody To Love - Extended Mix,Sigma,drum-and-bass,0.940756
8,Cinnamon - Hybrid Minds Remix,Hybrid Minds,drum-and-bass,0.938200
9,Crush,Pendulum,drum-and-bass,0.936757


## 10. Test Recommendations

Let's try the recommendation system with a few different songs and see what it comes up with.



In [13]:

recommend_song("I'm Yours", 10)

,track_name,artists,track_genre,score
0,Lucky,Jason Mraz;Colbie Caillat,acoustic,0.955732
1,Yakap,Zack Tabudlo,acoustic,0.901303
2,Can't Go Back Now,The Weepies;Deb Talan;Steve Tannen,acoustic,0.885698
3,Have It All,Jason Mraz,acoustic,0.883579
4,Stand Your Ground,Joshua Hyslop,acoustic,0.879552
5,Rain,Motohiro Hata,acoustic,0.872590
6,Better Together,Us The Duo,acoustic,0.870692
7,Come On Get Higher,Matt Nathanson,acoustic,0.870132
8,Blister In The Sun,Violent Femmes,acoustic,0.866513
9,You're the Sea,Andrew Belle,acoustic,0.865643


## 11. Save Processed Data

The preprocessing takes some time, so I'll save the processed data and feature matrix as pickle files.

The Streamlit app can load these files directly instead of doing all the preprocessing again.



In [14]:
import pickle

pickle.dump(songs, open('songs.pkl', 'wb'))
pickle.dump(X, open('X.pkl', 'wb'))

In [15]:
!pip install spotipy

Defaulting to user installation because normal site-packages is not writeable


In [16]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
from dotenv import load_dotenv
import os

load_dotenv()

client_id = os.getenv("SPOTIFY_CLIENT_ID")
client_secret = os.getenv("SPOTIFY_CLIENT_SECRET")


sp = spotipy.Spotify(
    auth_manager=SpotifyClientCredentials(
        client_id=client_id,
        client_secret=client_secret
    )
)

print("Spotify connected!")

Spotify connected!


In [17]:
import requests

In [18]:
def get_cover_url(track_name, artist):
    try:
        url = "https://itunes.apple.com/search"

        params = {
            "term": f"{track_name} {artist}",
            "media": "music",
            "entity": "song",
            "limit": 5
        }

        response = requests.get(url, params=params, timeout=10)

        if response.status_code != 200:
            return None

        results = response.json()["results"]

        if len(results) == 0:
            return None

        # Take the first result
        cover_url = results[0].get("artworkUrl100")

        if cover_url:
            # Get a larger image
            cover_url = cover_url.replace("100x100", "600x600")

        return cover_url

    except Exception as e:
        print(f"Error for {track_name} - {artist}: {e}")
        return None

In [19]:
cover = get_cover_url("Rait Zara Si", "A.R. Rahman")

print(cover)

https://is1-ssl.mzstatic.com/image/thumb/Music126/v4/6b/2c/76/6b2c7678-8423-0fc2-0602-05fa196e79b5/8903431857637_cover.jpg/600x600bb.jpg


In [20]:
print(type(songs))

<class 'pandas.core.frame.DataFrame'>


In [21]:
songs.head()

,track_name,artists,track_genre,danceability,popularity,energy,speechiness,acousticness,instrumentalness,liveness,valence,clean_title
0,Comedy,Gen Hoshino,acoustic,0.676,73,0.4610,0.1430,0.0322,0.000001,0.3580,0.715,comedy
1,Ghost - Acoustic,Ben Woodward,acoustic,0.420,55,0.1660,0.0763,0.9240,0.000006,0.1010,0.267,ghost - acoustic
2,To Begin Again,Ingrid Michaelson;ZAYN,acoustic,0.438,57,0.3590,0.0557,0.2100,0.000000,0.1170,0.120,to begin again
3,Can't Help Falling In Love,Kina Grannis,acoustic,0.266,71,0.0596,0.0363,0.9050,0.000071,0.1320,0.143,can't help falling in love
4,Hold On,Chord Overstreet,acoustic,0.618,82,0.4430,0.0526,0.4690,0.000000,0.0829,0.167,hold on


In [22]:
from IPython.display import Image, display

cover = get_cover_url("Believer", "Imagine Dragons")

print("Cover URL:", cover)

if cover:
    display(Image(url=cover))
else:
    print("No cover found")

Cover URL: https://is1-ssl.mzstatic.com/image/thumb/Music126/v4/11/7a/b8/117ab805-6811-8929-18b9-0fad7baf0c25/17UMGIM98210.rgb.jpg/600x600bb.jpg


In [23]:
song = "Rait Zara Si"
artist = "A.R. Rahman"

cover = get_cover_url(song, artist)

if cover:
    display(Image(url=cover))
else:
    print("No cover found")

In [24]:
result = recommend_song("Rait Zara Si", 10)
print(result)

                                        track_name  \
0                                          Humdard   
1                                           Fitoor   
2  Darkhaast (feat. Arijit Singh, Sunidhi Chauhan)   
3                                Aasan Nahin Yahan   
4                                        Tum Hi Ho   
5  Baatein Ye Kabhi Na (From "Khamoshiyan") - Male   
6                 Khamoshiyan (From "Khamoshiyan")   
7                         Phir Bhi Tumko Chaahunga   
8                  Thodi Jagah (From "Marjaavaan")   
9                             Muskurane - Romantic   

                                           artists track_genre     score  
0                                     Arijit Singh    pop-film  0.996156  
1  Mithoon;Arijit Singh;Neeti Mohan;Karan Malhotra    pop-film  0.994197  
2             Mithoon;Arijit Singh;Sunidhi Chauhan    pop-film  0.989518  
3                                     Arijit Singh    pop-film  0.987199  
4                             

In [25]:
recommend_song("Rait Zara Si", 10)

,track_name,artists,track_genre,score
0,Humdard,Arijit Singh,pop-film,0.996156
1,Fitoor,Mithoon;Arijit Singh;Neeti Mohan;Karan Malhotra,pop-film,0.994197
2,"Darkhaast (feat. Arijit Singh, Sunidhi Chauhan)",Mithoon;Arijit Singh;Sunidhi Chauhan,pop-film,0.989518
3,Aasan Nahin Yahan,Arijit Singh,pop-film,0.987199
4,Tum Hi Ho,Arijit Singh,pop-film,0.986881
5,"Baatein Ye Kabhi Na (From ""Khamoshiyan"") - Male",Jeet Gannguli;Arijit Singh,pop-film,0.984446
6,"Khamoshiyan (From ""Khamoshiyan"")",Jeet Gannguli;Arijit Singh,pop-film,0.982638
7,Phir Bhi Tumko Chaahunga,Mithoon;Arijit Singh;Shashaa Tirupati,pop-film,0.980172
8,"Thodi Jagah (From ""Marjaavaan"")",Arijit Singh;Tanishk Bagchi,pop-film,0.979913
9,Muskurane - Romantic,Jeet Gannguli;Arijit Singh,pop-film,0.978984


In [26]:
from IPython.display import Image, display

for _, row in result.iterrows():
    print(f"{row['track_name']} — {row['artists']}")
    
    cover_url = get_cover_url(row['track_name'], row['artists'])
    
    if cover_url:
        display(Image(url=cover_url, width=150))
    else:
        print("Cover not found")
    
    print("-" * 50)

Humdard — Arijit Singh


--------------------------------------------------
Fitoor — Mithoon;Arijit Singh;Neeti Mohan;Karan Malhotra


--------------------------------------------------
Darkhaast (feat. Arijit Singh, Sunidhi Chauhan) — Mithoon;Arijit Singh;Sunidhi Chauhan


--------------------------------------------------
Aasan Nahin Yahan — Arijit Singh


--------------------------------------------------
Tum Hi Ho — Arijit Singh


...

--------------------------------------------------
Khamoshiyan (From "Khamoshiyan") — Jeet Gannguli;Arijit Singh


--------------------------------------------------
Phir Bhi Tumko Chaahunga — Mithoon;Arijit Singh;Shashaa Tirupati


--------------------------------------------------
Thodi Jagah (From "Marjaavaan") — Arijit Singh;Tanishk Bagchi


--------------------------------------------------
Muskurane - Romantic — Jeet Gannguli;Arijit Singh


--------------------------------------------------


## Conclusion

The recommendation system is now able to take a song and find other songs that are similar to it using Spotify's audio features and genre information.

The processed files can then be used by the Streamlit app to generate recommendations.
